# PatchCore Lite Optuna

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


In [2]:
INSTALL_DEPS = False

if INSTALL_DEPS:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])


In [3]:
from src.common.config import load_config
from src.common.data import SpacepressoDataModule
from src.common.paths import resolve_path
from src.common.ranking import load_rankings
from src.common.seed import set_seed
from src.common.tuning import OptunaTuner, suggest_patchcore_lite_config


In [4]:
config = load_config(ROOT / "configs/patchcore_lite/juan_tuning.yaml")

local_data = ROOT / "data" / "spacepresso"
if local_data.exists():
    config["data"]["root"] = str(local_data)

config["data"]["load_images"] = False
config["tuning"]["output_dir"] = str(resolve_path(config["tuning"]["output_dir"], ROOT))
config["ranking"]["output_path"] = str(resolve_path(config["ranking"]["output_path"], ROOT))

set_seed(config.get("seed", 42))
config


{'seed': 42,
 'experiment': {'name': 'patchcore_lite_tuning',
  'owner': 'juan',
  'output_dir': 'outputs/juan/patchcore_lite_tuning'},
 'data': {'root': 'C:\\Users\\sjuan\\Desktop\\Spacespresso\\data\\spacepresso',
  'image_size': 224,
  'load_images': False},
 'method': {'name': 'patchcore_lite',
  'backbone': 'wide_resnet50_2',
  'batch_size': 4,
  'out_indices': [2, 3],
  'coreset_fraction': 0.01,
  'candidate_pool_size': 6000,
  'max_coreset_size': 1000,
  'projection_dim': 512,
  'bank_chunk_size': 2048,
  'sigma': 4.0},
 'training': {'enabled': True, 'mode': 'fit_memory_bank'},
 'tuning': {'study_name': 'patchcore_lite_juan',
  'direction': 'maximize',
  'metric': 'pixel_ap',
  'n_trials': 10,
  'n_splits': 5,
  'folds': [0],
  'good_fraction': 0.25,
  'anomaly_fraction': 1.0,
  'output_dir': 'C:\\Users\\sjuan\\Desktop\\Spacespresso\\outputs\\juan\\patchcore_lite_tuning\\optuna',
  'search_space': {'coreset_fraction': {'low': 0.002,
    'high': 0.02,
    'log': True},
   'max_co

In [5]:
dm = SpacepressoDataModule(**config["data"])
train_good = dm.load_train_good()
train_anomalies = dm.load_train_anomalies()

print("train_good:", len(train_good))
print("train_anomalies:", len(train_anomalies))

assert train_good
assert train_anomalies


train_good: 19005
train_anomalies: 235


In [6]:
tuner = OptunaTuner(
    base_config=config,
    train_good=train_good,
    train_anomalies=train_anomalies,
    suggest_config=suggest_patchcore_lite_config,
)

study = tuner.run()
study.best_value, study.best_params


[I 2026-05-07 14:46:56,473] A new study created in memory with name: patchcore_lite_juan


Using PatchCore backbone: wide_resnet50_2
Fitting PatchCore memory bank for class_01: 2080 images


PatchCore feature extraction:   0%|          | 0/520 [00:00<?, ?it/s]

Greedy coreset:   0%|          | 0/749 [00:00<?, ?it/s]

Seen bank: 1,630,720 patches x 1536 dims; candidate pool: 5,000 patches; coreset: 750 patches
Fitting PatchCore memory bank for class_02: 1705 images


...

PatchCore inference:   0%|          | 0/35 [00:00<?, ?it/s]

PatchCore inference:   0%|          | 0/42 [00:00<?, ?it/s]

PatchCore inference:   0%|          | 0/38 [00:00<?, ?it/s]

[I 2026-05-07 15:10:57,199] Trial 9 finished with value: 0.3148686267538668 and parameters: {'coreset_fraction': 0.004786458626331876, 'max_coreset_size': 1500, 'candidate_pool_size': 5000, 'projection_dim': 512, 'sigma': 2.4756379753499664, 'out_indices': '2,3'}. Best is trial 4 with value: 0.32226677617579425.


(0.32226677617579425,
 {'coreset_fraction': 0.0071075781167154115,
  'max_coreset_size': 2000,
  'candidate_pool_size': 4000,
  'projection_dim': 128,
  'sigma': 5.3829843923343,
  'out_indices': '2,3'})

In [7]:
rankings = load_rankings(ROOT)
rankings.head(20)


,rank,model_id,experiment_name,parent_experiment,owner,method,source,metric,score,pixel_ap,...,n_images,n_anomaly_pixels,validation_n_splits,validation_folds,validation_good_fraction,validation_anomaly_fraction,config_path,params_json,notes,created_at
0,1,patchcore_lite_baseline,patchcore_lite_baseline,NaN,juan,patchcore_lite,baseline,pixel_ap,0.322563,0.322563,...,1190,1217489,5,0,0.25,1.0,configs\patchcore_lite\juan_baseline.yaml,"{""backbone"": ""wide_resnet50_2"", ""bank_chunk_si...",PatchCore Lite baseline validation,2026-05-07T12:46:02.757469+00:00
1,2,patchcore_lite_juan_trial_0004,patchcore_lite_juan_trial_0004,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.322267,0.322267,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 4000, ""coreset_fractio...",Optuna trial 4,2026-05-07T13:10:57.233644+00:00
2,3,patchcore_lite_juan_trial_0009,patchcore_lite_juan_trial_0009,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.314869,0.314869,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 5000, ""coreset_fractio...",Optuna trial 9,2026-05-07T13:10:57.233644+00:00
3,4,patchcore_lite_juan_trial_0003,patchcore_lite_juan_trial_0003,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.313526,0.313526,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 6000, ""coreset_fractio...",Optuna trial 3,2026-05-07T13:10:57.233644+00:00
4,5,patchcore_lite_juan_trial_0006,patchcore_lite_juan_trial_0006,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.312060,0.312060,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 4000, ""coreset_fractio...",Optuna trial 6,2026-05-07T13:10:57.233644+00:00
5,6,patchcore_lite_juan_trial_0007,patchcore_lite_juan_trial_0007,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.308948,0.308948,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 5000, ""coreset_fractio...",Optuna trial 7,2026-05-07T13:10:57.233644+00:00
6,7,patchcore_lite_juan_trial_0005,patchcore_lite_juan_trial_0005,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.305698,0.305698,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 4000, ""coreset_fractio...",Optuna trial 5,2026-05-07T13:10:57.233644+00:00
7,8,patchcore_lite_juan_trial_0001,patchcore_lite_juan_trial_0001,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.304889,0.304889,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 5000, ""coreset_fractio...",Optuna trial 1,2026-05-07T13:10:57.233644+00:00
8,9,patchcore_lite_juan_trial_0008,patchcore_lite_juan_trial_0008,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.304730,0.304730,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 2000, ""coreset_fractio...",Optuna trial 8,2026-05-07T13:10:57.233644+00:00
9,10,patchcore_lite_juan_trial_0000,patchcore_lite_juan_trial_0000,patchcore_lite_tuning,juan,patchcore_lite,optuna,pixel_ap,0.303279,0.303279,...,1190,1217489,5,0,0.25,1.0,C:\Users\sjuan\Desktop\Spacespresso\outputs\ju...,"{""candidate_pool_size"": 5000, ""coreset_fractio...",Optuna trial 0,2026-05-07T13:10:57.233644+00:00
